## 1. Preprocessing Objective

This notebook prepares the cleaned dataset for modeling: encoding categorical features, imputing missing values, removing non-predictive columns, and splitting the data into training, validation, and test sets while preserving the class balance established in the EDA.

## 2. Import libraries and load raw dataset


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'Churn.csv'

df = pd.read_csv(data_path)

## 3. Separate the target `Exited` from the features

In [3]:
# Separate target from features
features = df.drop(columns=['Exited'])
target = df['Exited']

## 4. Remove `RowNumber`, `CustomerId`, and `Surname`.

In [4]:
# Drop identifier/unusable columns (flagged in EDA Section 2, finalized in Section 5)
features = features.drop(columns=['CustomerId', 'Surname'])


## 5. Split the raw data


In [5]:
# Assign to conventional ML notation for the split step
X = features
y = target

In [6]:
# Step 1: split off the test set (20% of the data)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

# Step 2: split the remaining 80% into 60% train and 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25, # 0.25 x 0.8 = 0.2
    stratify=y_temp,
    random_state=42
)

In [7]:
# Split sizes and class balance sanity check
print(f"Train: {X_train.shape[0]} rows, ({X_train.shape[0]/len(X):.1%})")
print(f"Validation: {X_val.shape[0]} rows, ({X_val.shape[0]/len(X):.1%})")
print(f"Test: {X_test.shape[0]} rows, ({X_test.shape[0]/len(X):.1%})")

Train: 6000 rows, (60.0%)
Validation: 2000 rows, (20.0%)
Test: 2000 rows, (20.0%)


## 6. Learn all preprocessing steps from the training set only

In [8]:
# 1. Median Imputation for Tenure
median_tenure = X_train['Tenure'].median()

X_train['Tenure'] = X_train['Tenure'].fillna(median_tenure)
X_val['Tenure'] = X_val['Tenure'].fillna(median_tenure)
X_test['Tenure'] = X_test['Tenure'].fillna(median_tenure)

In [9]:
# 2. One-hot encoding for Geography and Gender
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[['Geography', 'Gender']]) # fit on training data only

# transform the training, validation, and test sets using what was learned from the training data
train_encoded = encoder.transform(X_train[['Geography', 'Gender']])
val_encoded = encoder.transform(X_val[['Geography', 'Gender']])
test_encoded = encoder.transform(X_test[['Geography', 'Gender']])

In [10]:
# 3. Scale numerical features (CreditScore, Age, Tenure, Balance, NumOfProducts, EstimatedSalary)
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

scaler = StandardScaler()
scaler.fit(X_train[numerical_cols]) # learn mean and std from training data only

X_train[numerical_cols] = scaler.transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

##### # 4. Pass binary features throught unchanged
`HasCrCard` and `IsActiveMember` are already 0/1, no imputation, encoding, or scaling needed.

### 7. Verify that the preprocessing steps have been applied correctly

In [11]:
# 1. No missing values
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0
0


In [12]:
# 2. Dataset dimensions are correct
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

X_train: (6000, 11)
X_val: (2000, 11)
X_test: (2000, 11)


In [13]:
# 3. Class proportions remain similar across splits
print(f"Churn rate in training set: {y_train.mean():.4f}")
print(f"Churn rate in validation set: {y_val.mean():.4f}")
print(f"Churn rate in test set: {y_test.mean():.4f}")

Churn rate in training set: 0.2038
Churn rate in validation set: 0.2035
Churn rate in test set: 0.2035


In [14]:
# 4. Training/validation/test records do not overlap

train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

print(f"Train/Val overlap: {len(train_idx & val_idx)}")
print(f"Train/Test overlap: {len(train_idx & test_idx)}")
print(f"Val/Test overlap: {len(val_idx & test_idx)}")

Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0


## 8. Save Processed Data

The final train, validation, and test splits are saved to `data/processed/` so they can be loaded directly in the modeling notebook without re-running preprocessing.

In [15]:
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

In [16]:
X_train.to_csv(processed_dir / 'X_train.csv', index=False)
X_val.to_csv(processed_dir / 'X_val.csv', index=False)
X_test.to_csv(processed_dir / 'X_test.csv', index=False)

y_train.to_csv(processed_dir / 'y_train.csv', index=False)
y_val.to_csv(processed_dir / 'y_val.csv', index=False)
y_test.to_csv(processed_dir / 'y_test.csv', index=False)